In [1]:
# !pip install ipython==8.1.0

In [2]:
# # Comment the following if you are running your code locally
# # This mounts your Google Drive to the Colab VM.
# from google.colab import drive
# drive.mount('/content/drive')
# drive_folder = '/content/drive/MyDrive'
# notebook_folder = drive_folder + '/project'
# %cd {notebook_folder}

# You don't need to run unless the csv files are missing
# %cd ./cs682/data
# !bash prepare.sh
# %cd ../../

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from transformers import get_linear_schedule_with_warmup
import json
from datasets import load_dataset

from cs682.models.student import FunnelTransformer
from cs682.evaluator import evaluate
from tqdm import tqdm

In [7]:
# ds = load_dataset("glue", "mnli")
# print(ds["train"][0])
# {'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.', 'hypothesis': 'Product and geography are what make cream skimming work. ', 'label': 1, 'idx': 0}

In [26]:
class TrainConfig:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def __repr__(self):
        return f"TrainConfig({self.__dict__})"

def check_accuracy(
    model: FunnelTransformer, loader: DataLoader, device, max_batches: int = None
):
    model.eval()
    correct = 0
    total = 0
    use_amp = device.type == "cuda"
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["labels"].to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
            predicted = torch.argmax(out["logits"], dim=1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    return correct / total


def train(config: TrainConfig, train_loader: DataLoader, validation_loader: DataLoader):
    print(config)

    criterion = config.criterion
    optimizer = config.optimizer
    scheduler = config.scheduler
    max_grad_norm = config.max_grad_norm

    epochs = config.epochs
    log_every_k = config.log_every_k
    accuracy_every_k = config.accuracy_every_k

    model = config.model

    device = config.device
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)

    model.to(device)

    losses = []
    train_accuracies = []
    validation_accuracies = []
    for epoch in range(epochs):
        model.train()
        for i, batch in enumerate(
            tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
        ):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(out["logits"], labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(loss.item())

            if i % log_every_k == 0:
                print(
                    f"Epoch {epoch + 1} | Step {i} | "
                    f"loss={loss:.4f} | "
                )

            if i % accuracy_every_k == 0:
                train_acc = check_accuracy(
                    model, train_loader, device, max_batches=20
                )
                val_acc = check_accuracy(model, validation_loader, device)
                train_accuracies.append(train_acc)
                validation_accuracies.append(val_acc)
                print(f"  Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")
                model.train()

    return model, losses, train_accuracies, validation_accuracies

def make_collate_fn(tokenizer, max_length):
    def collate_fn(batch):
        premises = [item["premise"] for item in batch]
        hypotheses = [item["hypothesis"] for item in batch]
        labels = [item["label"] for item in batch]
        enc = tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc

    return collate_fn

In [ ]:
dataset = "yelp"
is_two_classes = True
num_classes = 2 if is_two_classes else 5

model_checkpoint_path = f"./models/student_{dataset}_{is_two_classes}.pt"

model_save_path = f"./models/student_MNLI_{dataset}_{is_two_classes}.pt"
train_save_path = f"./models/student_MNLI_{dataset}_{is_two_classes}.json"

epochs = 3
learning_rate = 3e-5
weight_decay = 1e-4
warmup_ratio = 0.1
max_grad_norm = 1.0
batch_size = 32
num_workers = 4
log_every_k = 500
accuracy_every_k = 1000
max_length = 128
dropout = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
checkpoint = torch.load(model_checkpoint_path, map_location=device)
model, tokenizer = FunnelTransformer.from_bert(
    block_layers=checkpoint["args"]["block_layers"],
    num_classes=num_classes,
    dropout=checkpoint["args"]["dropout"],
)
model.load_state_dict(checkpoint["model_state_dict"])

# Load dataset and load to data loader
ds = load_dataset("glue", "mnli")


# Replace head with 3 class
hidden_d = model.cls_head[0].in_features
model.cls_head = nn.Sequential(
    nn.Linear(hidden_d, hidden_d),
    nn.Tanh(),
    nn.Dropout(dropout),
    nn.Linear(hidden_d, 3),  # MNLI classes
)

# Filter out examples with label == -1 (unlabeled)
train_ds = ds["train"].filter(lambda x: x["label"] != -1)
val_ds = ds["validation_matched"].filter(lambda x: x["label"] != -1)

collate_fn = make_collate_fn(tokenizer, max_length)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=True,
)

validation_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=True,
)

print(f"Train: {len(train_ds)} | Validation: {len(val_ds)}")

optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay
)
total_steps = len(train_loader) * epochs
warmup_steps = int(total_steps * warmup_ratio)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

train_config = TrainConfig(
    model=model,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
    log_every_k=log_every_k,
    accuracy_every_k=accuracy_every_k,
    max_grad_norm=max_grad_norm,
    device=device,
)

model, losses, train_accs, val_accs = train(train_config, train_loader, validation_loader)

with open(train_save_path, "w") as f:
    obj = {
        "losses": losses,
        "train_accs": train_accs,
        "validation_accs": val_accs
    }

    json.dump(obj, f, indent=2)

train_config_dict = {
    "dataset": dataset,
    "is_two_classes": is_two_classes,
    "epochs": epochs,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "warmup_ratio": warmup_ratio,
    "max_grad_norm": max_grad_norm,
    "batch_size": batch_size,
    "num_workers": num_workers,
    "log_every_k": log_every_k,
    "accuracy_every_k": accuracy_every_k,
    "max_length": max_length,
    "dropout": dropout,
}

torch.save({
    "model_state_dict": model.state_dict(),
    "args": train_config_dict,
}, model_save_path)
print(f"\nModel saved to {model_save_path}")

Loading 'google/bert_uncased_L-8_H-512_A-8' for embedding initialisation ...
  Copied token embeddings  (30522 x 512)
  Copied positional embeddings (first 512 positions)
  Block config: [2, 2, 2]  |  num_classes: 2
Train: 392702 | Validation: 9815
TrainConfig({'model': FunnelTransformer(
  (token_emb): Embedding(30522, 512, padding_idx=0)
  (pos_emb): Embedding(512, 512)
  (emb_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (emb_drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-2): 3 x FunnelBlock(
      (layers): ModuleList(
        (0-1): 2 x TransformerLayer(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (ffn): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.1, inplace=False)
            (3): Linear(in_features=2048, out_features=

Epoch 1/1:   0%|          | 0/12272 [00:00<?, ?it/s]

Epoch 1 | Step 0 | loss=1.0889 | 


Epoch 1/1:   0%|          | 3/12272 [00:11<10:02:56,  2.95s/it]

  Train acc: 0.3484 | Val acc: 0.3406


Epoch 1/1:   0%|          | 4/12272 [00:12<10:22:36,  3.05s/it]



Model saved to ./models/student_MNLI_yelp_True.pt


In [18]:
test_ds = ds["test_matched"].filter(lambda x: x["label"] != -1)


test_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=num_workers,
    pin_memory=True,
)

evaluate(model, test_loader, 3, device)

Filter: 100%|██████████| 9796/9796 [00:00<00:00, 274637.39 examples/s]

Starting evaluation.


0 / 12272 Complete
50 / 12272 Complete
100 / 12272 Complete
150 / 12272 Complete
200 / 12272 Complete
250 / 12272 Complete
300 / 12272 Complete
350 / 12272 Complete
400 / 12272 Complete
450 / 12272 Complete
500 / 12272 Complete
550 / 12272 Complete
600 / 12272 Complete
650 / 12272 Complete
700 / 12272 Complete
750 / 12272 Complete
800 / 12272 Complete
850 / 12272 Complete
900 / 12272 Complete
950 / 12272 Complete
1000 / 12272 Complete
1050 / 12272 Complete
1100 / 12272 Complete
1150 / 12272 Complete
1200 / 12272 Complete
1250 / 12272 Complete
1300 / 12272 Complete
1350 / 12272 Complete
1400 / 12272 Complete
1450 / 12272 Complete
1500 / 12272 Complete
1550 / 12272 Complete
1600 / 12272 Complete
1650 / 12272 Complete
1700 / 12272 Complete
1750 / 12272 Complete
1800 / 12272 Complete
1850 / 12272 Complete
1900 / 12272 Complete
1950 / 12272 Complete
2000 / 12272 Complete
2050 / 12272 Complete
2100 / 12272 Complete
2150 / 12272 Complete
2200 / 12272 Complete
2250 / 12272 Complete
2300 / 1227

(0.6995864548690864, 0.30041354513091356)